# Suite2p Pipeline for Two-Photon Calcium Imaging

This notebook runs the full [suite2p](https://github.com/MouseLand/suite2p) pipeline.  
Every parameter from `default_ops()` is listed and documented — including **hidden ones not in the official docs** (marked ⚠️).

## `ops` vs `settings` — what's the difference?

| | `ops` | `settings` |
|--|--|--|
| **What it is** | One flat dict with all params | Nested dict split by stage (`settings["detection"]`, `settings["registration"]`, …) |
| **How to get it** | `default_ops()` | `default_settings()` |
| **Who uses it** | `run_s2p()` — the standard pipeline | Individual stage wrappers (`detection_wrapper()`, `registration_wrapper()`, …) |
| **Who should use it** | **You, in this notebook** | Advanced users calling stages directly |

In short: **use `ops`** for the standard pipeline. `settings` is the internal representation the new modular API uses.

**Reference:** Pachitariu et al. 2017 | Docs: https://suite2p.readthedocs.io | Code: https://github.com/MouseLand/suite2p

In [ ]:
# %pip install suite2p

import suite2p
from suite2p import run_s2p, default_ops
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

print(f"suite2p version: {suite2p.__version__}")

## 1. Data paths

In [ ]:
DATA_PATH = [r"/path/to/your/raw/data"]
SAVE_PATH = r"/path/to/your/processed"
FAST_DISK = r""   # fast SSD scratch; "" = same as SAVE_PATH

for p in DATA_PATH:
    assert Path(p).exists(), f"Data path not found: {p}"
Path(SAVE_PATH).mkdir(parents=True, exist_ok=True)
print("Paths OK")

## 2. Build the ops dictionary

Every key from `default_ops()` is listed below.  
Keys marked **⚠️ HIDDEN** are not in the official documentation but are present in the source code.

In [ ]:
ops = default_ops()
print(f"{len(ops)} parameters in default_ops()")

### 2.1 Paths & File I/O

In [ ]:
# ── Standard paths ────────────────────────────────────────────────────────────
ops["data_path"]  = DATA_PATH
ops["save_path0"] = SAVE_PATH   # suite2p creates a 'suite2p' subdirectory here
ops["fast_disk"]  = FAST_DISK

# Optional: name of the results subdirectory inside save_path0.
# Default: [] (suite2p uses 'suite2p').
ops["save_folder"] = []

# Whether to also search one level of subdirectories inside each data_path.
# Default: False.
ops["look_one_level_down"] = False

# Optional: restrict the search to these named child folders only.
# Leave as [] to search all subfolders.  Default: [].
ops["subfolders"] = []

# ── File format flags ─────────────────────────────────────────────────────────

# Whether to use scikit-image (sklearn.external.tifffile) for TIFF reading
# instead of the default ScanImage TIFF reader.  Try True if the default
# reader fails to open your files.
# ⚠️ HIDDEN — not in official docs.  Default: False.
ops["force_sktiff"] = False

# Whether to process Bruker Prairie View single-page TIFFs.
# suite2p will split files by channel based on filenames ('Ch1' = channel 1).
# Default: False.
ops["bruker"] = False

# Bruker multiplane ordering: True = bidirectional (0,1,2,2,1,0),
# False = sequential (0,1,2,0,1,2).  Only relevant when bruker=True.
# ⚠️ HIDDEN — not in official docs.  Default: False.
ops["bruker_bidirectional"] = False

# Whether to read ScanImage mesoscope files (special multi-ROI format).
# Default: False.
ops["mesoscan"] = False

# ── HDF5 / NWB inputs ─────────────────────────────────────────────────────────

# List of HDF5 file paths. Leave as [] if file discovery should handle them.
ops["h5py"] = []

# Dataset key inside the HDF5 file that holds the imaging array.
ops["h5py_key"] = "data"

# Path to a NWB file (deactivates data_path discovery).
# ⚠️ HIDDEN — not in official docs.  Default: "".
ops["nwb_file"] = ""

# HDF5 driver for the NWB file (empty string = local file, no special driver).
# ⚠️ HIDDEN — not in official docs.  Default: "".
ops["nwb_driver"] = ""

# TwoPhotonSeries name inside the NWB file.  Defaults to the first series found.
# ⚠️ HIDDEN — not in official docs.  Default: "".
ops["nwb_series"] = ""

# ── Binary file handling ──────────────────────────────────────────────────────

# Whether to delete the intermediate binary after the full pipeline finishes.
# Keep False while tuning parameters so you can rerun without re-converting TIFFs.
# Default: False.
ops["delete_bin"] = False

# When fast_disk differs from save_path0, move (rather than copy) the binary
# from fast_disk to save_path0 when processing is complete.
# ⚠️ HIDDEN — not in official docs.  Default: False.
ops["move_bin"] = False

# ── Photostimulation / artifact frames ────────────────────────────────────────
# To exclude frames during ROI detection (e.g. photostimulation artefacts),
# save a NumPy integer array called 'bad_frames.npy' in ops['data_path'][0].

# ── Output formats ────────────────────────────────────────────────────────────

# Save results as a MATLAB Fall.mat file in addition to .npy files.
# Default: False.
ops["save_mat"] = False

# Export results to NWB format.  Default: False.
ops["save_NWB"] = False

### 2.2 Acquisition / microscope

In [ ]:
# ── Frame rate ────────────────────────────────────────────────────────────────

# Volume rate PER PLANE in Hz.  For multi-plane recordings divide the volume
# rate by the number of planes.  Example: 30-Hz volume, 3 planes → fs = 10.
# Default: 10.0.  ← This is used for deconvolution kernel scaling.
ops["fs"] = 30.0   # ← set to your actual per-plane frame rate

# ── Planes & channels ─────────────────────────────────────────────────────────

# Number of imaging planes interleaved in the data files.  1 = single-plane.
# Default: 1.
#
# 💾 DATA TYPE — ScanImage saves raw TIFFs as int16 (2 bytes/pixel).
#   Suite2p converts every frame to float32 (4 bytes/pixel) when writing
#   data.bin.  This doubles the per-frame size relative to the TIFF files
#   sitting on disk:
#
#     Format              512×512 frame    1000×2000 frame
#     ────────────────── ─────────────── ─────────────────
#     Raw TIFF  (int16)      0.5 MB/frame      4 MB/frame
#     data.bin  (float32)    1.0 MB/frame      8 MB/frame
#
#   All disk and RAM estimates throughout this notebook use the float32 sizes.
#
# 💾 DISK IMPACT — multiplies all binary files by nplanes.
#   Each plane gets its own data.bin.  RAM impact is per-plane (processed
#   one at a time), so nplanes does NOT multiply peak RAM.
#   data.bin size per plane (20 min @ 30 Hz = 36 000 frames):
#     512×512  :  36 000 × 1 MB =  36 GB / plane
#     1000×2000:  36 000 × 8 MB = 288 GB / plane
#   nplanes = 4 on 1000×2000 → ~1.15 TB total on disk.  Ensure you have space.
ops["nplanes"] = 1

# Number of channels per plane.  Data must be channel-interleaved:
# (t0,ch1), (t0,ch2), (t1,ch1), (t1,ch2), …
# Default: 1.
#
# 💾 DISK IMPACT — doubles all binary files when nchannels=2.
#   Both channels are written to data.bin; during registration only the
#   functional channel is loaded, so batch_size RAM is unchanged.
#   Extra disk per plane (20 min @ 30 Hz):
#     512×512  :  +36 GB  (now 72 GB total)
#     1000×2000: +288 GB  (now 576 GB total)
ops["nchannels"] = 1

# Which channel (1-based) contains the functional calcium signal.
# Default: 1.
ops["functional_chan"] = 1

# Which channel (1-based) to use for motion registration.
# 1 = functional (default); 2 = anatomical.
# Using a stable structural channel can improve registration quality.
# Default: 1.
ops["align_by_chan"] = 1

# ── Timing & frame control ────────────────────────────────────────────────────

# If > 0, process only this many frames per plane.  Useful for quick tests.
# Default: -1 (all frames).
ops["frames_include"] = -1

# List of flyback frame indices to ignore at the start of each file.
# Some microscopes write garbage frames during PMT/scanner settling.
# ⚠️ HIDDEN — not in official docs.  Default: [].
ops["ignore_flyback"] = []

# ── Bidirectional scanning correction ─────────────────────────────────────────

# Whether to estimate and correct the phase offset between odd and even scan
# lines (bidirectional resonant scanning artefact).  suite2p estimates the
# offset from the first nimg_init frames and applies it to all subsequent frames.
# Default: False.
ops["do_bidiphase"] = False

# Fixed bidirectional phase offset in pixels.  Any non-zero value is applied
# directly, overriding do_bidiphase estimation.
# Default: 0.
ops["bidiphase"] = 0

# Whether bidirectional correction has already been applied to the data
# (e.g. by the acquisition software).  When True, suite2p skips correction.
# ⚠️ HIDDEN — not in official docs.  Default: False.
ops["bidi_corrected"] = False

# ── Multi-plane parallel processing ───────────────────────────────────────────

# Whether to run planes in parallel on a compute cluster / multi-GPU server.
# ⚠️ HIDDEN — experimental.  Default: False.
ops["multiplane_parallel"] = False

# ── GUI display ───────────────────────────────────────────────────────────────

# Pixel aspect ratio: µm per pixel in X / µm per pixel in Y.
# Used only by the GUI for correct display aspect ratio.  Default: 1.0.
ops["aspect"] = 1.0

### 2.3 Registration (motion correction)

In [ ]:
# ── Master switch ─────────────────────────────────────────────────────────────

# Whether to run registration.  Default: True.
ops["do_registration"] = True

# Whether to keep the unregistered raw binary on disk.  Needed to view raw vs.
# registered movies in the GUI and required for two_step_registration.
# Default: False.  (The official docs say True — that is WRONG; source is False.)
#
# 💾 DISK IMPACT — creates a second full copy of the binary (data_raw.bin).
#   Extra disk per plane (20 min @ 30 Hz = 36 000 frames):
#     512×512  :  +36 GB   (data.bin = 36 GB → total = 72 GB)
#     1000×2000: +288 GB   (data.bin = 288 GB → total = 576 GB)
#   Only enable if you need the GUI raw/registered comparison or two_step_registration.
ops["keep_movie_raw"] = False

# ── Reference image ───────────────────────────────────────────────────────────

# Number of initial frames used to build the reference image.  These frames are
# first quickly pre-aligned, then averaged.
# Default: 300.  (Official docs say 200 — that is WRONG; source is 300.)
#
# 💾 MEMORY IMPACT: LOW — safe to increase freely on either frame size.
#   ScanImage saves as int16 (2 bytes/pixel) → float32 (4 bytes/pixel) internally.
#   Frame sizes:                int16 (raw TIFF)    float32 (internal)
#     512×512                     0.5 MB/frame         1 MB/frame
#     1000×2000                   4.0 MB/frame         8 MB/frame
#
#   nimg_init frames loaded at once:
#     512×512,   nimg_init=300 : 300 ×  1 MB =  0.3 GB  (negligible)
#     1000×2000, nimg_init=300 : 300 ×  8 MB =  2.4 GB  (~3% of 96 GB — still fine)
#
#   GUIDANCE: use 300–1000.  More frames → better reference → better registration.
#   Diminishing returns beyond ~500.  Memory is NOT the limiting factor here.
ops["nimg_init"] = 300

# If True, and a reference image (ops['refImg']) is already stored in an ops
# file, skip computing a new one and use the stored image directly.
# ⚠️ HIDDEN — not in official docs.  Default: False.
ops["force_refImg"] = False

# ── Rigid registration ────────────────────────────────────────────────────────

# Number of frames processed per GPU/CPU registration batch.
# Default: 500.  (Official docs say 200 — that is WRONG; source is 500.)
#
# ⚠️ GPU VRAM IS THE REAL BOTTLENECK — not system RAM.
#   When a CUDA GPU is available, suite2p runs registration on the GPU.
#   Each batch must fit in VRAM.  The raw-frame + FFT buffers alone account for
#   ~2× frame_size per frame, but non-rigid registration (nonrigid=True, default)
#   adds per-block phase-correlation maps that dominate on large frames.
#
#   With nonrigid=True and block_size=[128,128], a 1000×2000 frame splits into
#   ~128 blocks.  Suite2p computes a correlation map per block per frame, plus
#   reference copies and PyTorch's allocator overhead.  The actual VRAM footprint
#   can be 10–30× higher than the naive "frames × frame_size" estimate, and it
#   scales with both batch_size and frame area.
#
#   Empirical guidance for a 12 GB GPU with nonrigid=True (block_size=[128,128]):
#
#     512×512 frames:                    1000×2000 frames:
#       batch_size =  500 →  safe ✓        batch_size = 500 → CRASH ✗
#       batch_size = 1000 →  safe ✓        batch_size = 100 → may crash ✗
#       batch_size = 1500 →  borderline ⚠️  batch_size =  50 → borderline ⚠️
#       batch_size = 2000 →  CRASH ✗       batch_size =  30 → safe ✓ (empirical)
#
#   GUIDANCE on a 12 GB GPU:
#     512×512  : keep default (500).  Can push to ~1200–1500.
#     1000×2000: start at 30 and increase until stable.  The optimal value
#                is dataset-dependent — nonrigid block count and frame content
#                both matter.  Start low and binary-search upward.
#
#   Two levers to relax the VRAM constraint WITHOUT reducing batch_size:
#     1. nonrigid=False  — eliminates block maps entirely; only 2–3 full-frame
#                          buffers needed → allows much larger batch_size.
#     2. block_size=[256,256] — cuts block count from ~128 to ~32 for 1000×2000
#                          → significantly less VRAM per frame.
#
#   NOTE: system RAM (96 GB) is NOT the constraint here.  VRAM is.
#   If you force CPU-only (ops["gpu_port"] = 0 or no CUDA), batch_size can be
#   hundreds–thousands for either frame size without crashing system RAM.
ops["batch_size"] = 500

# Maximum allowed rigid shift as a fraction of the frame dimension.
# The pixel limit applied is: max(Ly, Lx) * maxregshift.
# Default: 0.1 (10% of frame size).
ops["maxregshift"] = 0.1

# Gaussian smoothing sigma (pixels) applied to both the frame and the reference
# before computing the phase-correlation.  For 1P data at 512×512 the docs
# recommend > 4.  Default: 1.15.
ops["smooth_sigma"] = 1.15

# Gaussian smoothing sigma applied across the temporal dimension (frames) before
# phase-correlation.  Helps when individual frames have very low SNR.
# 0 = disabled.  Try 1–2 for noisy datasets.  Default: 0.
ops["smooth_sigma_time"] = 0

# Whether to normalize frames (zero-mean, unit-variance) before computing
# the phase-correlation.  Reduces bias from bright static structures.
# ⚠️ HIDDEN — not in official docs.  Default: True.
ops["norm_frames"] = True

# Threshold that determines which frames are considered 'bad' (high-motion)
# and excluded when computing the final crop region of the registered movie.
# LOWER values exclude MORE frames.  Stored as ops['badframes'] after registration.
# ⚠️ HIDDEN — not in official docs.  Default: 1.0.
ops["th_badframes"] = 1.0

# Subpixel resolution for the registration: 1/subpixel pixels.
# 10 = register to 0.1 pixel accuracy.  Default: 10.
ops["subpixel"] = 10

# Whether to pad the frame with zeros during the FFT shift step.
# Can reduce wrap-around artefacts at the frame edges.
# ⚠️ HIDDEN — not in official docs.  Default: False.
ops["pad_fft"] = False

# Run registration twice: first pass uses the original binary, second pass
# uses the output of the first to build a better reference image.
# Useful for low-SNR data.  Requires keep_movie_raw = True.  Default: False.
ops["two_step_registration"] = False

# ── Non-rigid registration ────────────────────────────────────────────────────

# Whether to perform non-rigid (block-wise) registration in addition to global
# rigid.  Non-rigid corrects local tissue deformations (cardiac pulse, z-drift).
# Default: True.
#
# ⚠️ GPU VRAM: setting this to False dramatically reduces VRAM usage during
#   registration.  With nonrigid=False, only 2–3 full-frame buffers are needed
#   per batch.  For 1000×2000 frames on a 12 GB GPU, consider nonrigid=False
#   if you cannot tolerate a very small batch_size, then inspect quality.
ops["nonrigid"] = True

# Block size [y, x] in pixels for non-rigid registration.  Powers of 2 (128,
# 256, 384) are recommended for efficient FFTs.  Smaller blocks → finer
# correction but slower and more noise-sensitive.  Default: [128, 128].
#
# ⚠️ GPU VRAM: smaller blocks = more blocks = more per-block FFT maps = more VRAM.
#   For 1000×2000: [128,128] → ~128 blocks; [256,256] → ~32 blocks (~4× less VRAM).
#   Try [256,256] if batch_size=30 is too slow and you need a larger batch.
ops["block_size"] = [128, 128]

# SNR threshold for accepting a block's shift estimate (peak-to-noise ratio of
# the phase-correlation map).  Blocks below this are interpolated from neighbors.
# For 1P data ~1.5.  Default: 1.2.
ops["snr_thresh"] = 1.2

# Maximum allowed block shift in pixels, relative to the global rigid shift.
# Prevents blocks from drifting far from the whole-frame solution.  Default: 5.
ops["maxregshiftNR"] = 5

# ── 1-photon mode ─────────────────────────────────────────────────────────────

# Enable high-pass spatial filtering and edge tapering before each registration
# step.  Removes slow background gradients that corrupt phase-correlation.
# Set True for 1P data; leave False for standard 2P.  Default: False.
ops["1Preg"] = False

# Window size in pixels for the spatial high-pass filter used in 1P mode
# (subtracts a boxcar-average of this width before computing shifts).
# Default: 42.  NOTE: in the source this key is 'spatial_hp_reg', not 'spatial_hp'.
ops["spatial_hp_reg"] = 42

# Gaussian smoothing sigma applied before the high-pass filter in 1P mode.
# 0 = no pre-smoothing.  Default: 0.
ops["pre_smooth"] = 0

# Pixels at each edge set to zero (tapered) before registration, to reduce
# vignetting artefacts.  Must be ≥ 3 × smooth_sigma.  Default: 40.
ops["spatial_taper"] = 40

# ── Save registered TIFFs ─────────────────────────────────────────────────────

# Write the registered functional channel to TIFF files on disk (large!).
# Default: False.
#
# 💾 DISK IMPACT — writes the full registered movie as uint16 TIFF (2 bytes/px).
#   20 min @ 30 Hz = 36 000 frames:
#     512×512  :  36 000 × 0.5 MB =  18 GB
#     1000×2000:  36 000 × 4.0 MB = 144 GB
ops["reg_tif"] = False

# Same for the second (anatomical) channel.  Default: False.
# 💾 Same disk impact as reg_tif above.
ops["reg_tif_chan2"] = False

### 2.4 ROI detection — suite2p (sparsery / sourcery)

In [ ]:
# ── Master switches ───────────────────────────────────────────────────────────

# Whether to run ROI detection.  Default: True.
ops["roidetect"] = True

# Whether to run spike deconvolution.  Default: True.
ops["spikedetect"] = True

# ── Detection algorithm ───────────────────────────────────────────────────────

# Use the sparse-mode (sparsery) cell detection algorithm.
# True (default) = sparsery, designed for sparsely active neurons.
# False = sourcery, the original dense algorithm (slower).
# NOTE: The official docs say default is False — that is WRONG; source is True.
ops["sparse_mode"] = True

# ── Spatial scale ─────────────────────────────────────────────────────────────

# Expected cell scale in pixels.  Controls the spatial filter used to find seeds.
# 0 = automatic multi-scale (recommended on first attempt).
# 1 ≈ 6 px | 2 ≈ 12 px | 3 ≈ 24 px | 4 ≈ 48 px.
# Default: 0.
ops["spatial_scale"] = 0

# ── Temporal binning ──────────────────────────────────────────────────────────

# Maximum number of binned frames used to compute the covariance matrix that
# drives ROI detection.  suite2p bins every B = n_frames / nbinned frames into
# one, then loads the ENTIRE binned movie into RAM at once for correlation-based
# cell finding.  This is the most dangerous parameter for memory.
# Default: 5000.
#
# ⚠️  MEMORY IMPACT: HIGH — this is THE parameter most likely to crash your machine.
#
#   After registration, suite2p stores data.bin as float32 (4 bytes/pixel).
#   The binned movie is held ENTIRELY in system RAM during detection (CPU step):
#     RAM = nbinned × Ly × Lx × 4 bytes
#
#   For 512×512 frames (1 MB/frame float32):
#     nbinned =  5 000 →   5 GB  ← DEFAULT  (fine on 96 GB, <6% of RAM)
#     nbinned = 10 000 →  10 GB             (fine on 96 GB)
#     nbinned = 20 000 →  20 GB             (fine on 96 GB)
#     nbinned = 40 000 →  40 GB             (fine on 96 GB, ~42%)
#     nbinned = 80 000 →  80 GB             (safe maximum on 96 GB)
#     nbinned = 96 000 →  96 GB  ← CRASH    (no headroom for OS + other processes)
#
#   For 1000×2000 frames (8 MB/frame float32):
#     nbinned =  5 000 →  40 GB  ← DEFAULT  (~42% of 96 GB — manageable)
#     nbinned =  8 000 →  64 GB             (~67% of 96 GB — tight but OK)
#     nbinned = 10 000 →  80 GB             (~83% of 96 GB — risky)
#     nbinned = 11 000 →  88 GB             (very risky — OOM likely with OS overhead)
#     nbinned = 12 000 →  96 GB  ← CRASH
#
#   GUIDANCE on a 96 GB machine:
#     512×512  : keep nbinned ≤ 80 000.  Default (5 000) is very conservative;
#                increasing to 20 000–40 000 can slightly improve detection
#                quality for short or low-SNR recordings.
#     1000×2000: the default (5 000) already uses 40 GB.  Keep it ≤ 8 000.
#                If you also enable denoise=True, halve those limits (see below).
#
#   IMPORTANT: nbinned is the BINNED frame count, not the raw frame count.
#   For a 20-min recording at 30 Hz = 36 000 raw frames with nbinned=5000:
#     binsize ≈ 7 → ~5 143 binned frames → memory stays ~40 GB regardless of
#     recording length.  Longer recordings just bin more frames per bin.
ops["nbinned"] = 5000

# ── Threshold ─────────────────────────────────────────────────────────────────

# Multiplier applied to the automatically determined activity threshold when
# searching for cell seeds.
# HIGHER → fewer, more confident ROIs.
# LOWER  → more ROIs, including small/dim cells, but also more false positives.
# Default: 1.0.  (Official docs say 5.0 — that is WRONG; source is 1.0.)
ops["threshold_scaling"] = 1.0

# ── Temporal baseline (high-pass) ─────────────────────────────────────────────

# Running-mean window in frames subtracted from each pixel's trace before
# correlation-based cell finding.  Removes slow bleaching/drift trends.
# For 1P data the docs recommend setting below 10.  Default: 100.
ops["high_pass"] = 100

# ── Spatial high-pass for neuropil subtraction during detection ───────────────

# Window (in pixels) for the uniform spatial high-pass filter applied to the
# binned movie BEFORE detection, to subtract local neuropil background.
# This is DIFFERENT from spatial_hp_reg (which is for registration).
# ⚠️ HIDDEN — not in official docs.  Default: 25.
ops["spatial_hp_detect"] = 25

# ── Morphological constraints ─────────────────────────────────────────────────

# Only keep pixels that are connected (8-connected) to the brightest pixel of
# each ROI.  Set False to allow fragmented ROIs (useful for dendrites/boutons).
# Default: True.
ops["connected"] = True

# Maximum fraction of a cell's pixels that may overlap another already-detected
# cell before the new ROI is discarded.  1.0 = accept all.  Default: 0.75.
ops["max_overlap"] = 0.75

# Maximum number of detection iterations (the algorithm stops earlier if no
# more peaks are above threshold).  Default: 20.
ops["max_iterations"] = 20

# Whether to denoise the binned movie with PCA before detection (sparse_mode only).
# Can improve detection in very noisy recordings but adds compute time.
# ⚠️ Not prominently documented.  Default: False.
#
# ⚠️  MEMORY IMPACT: HIGH — roughly DOUBLES the RAM of the detection step.
#   PCA denoising reshapes the binned movie into a 2D matrix and computes
#   the covariance; the working arrays are approximately the same size as
#   the binned movie itself.
#
#   For 512×512 frames (1 MB/frame):
#     nbinned= 5 000 → binned movie  5 GB → PCA adds  ~5 GB → total ~10 GB  ✓
#     nbinned=40 000 → binned movie 40 GB → PCA adds ~40 GB → total ~80 GB  ✓ (96 GB)
#     nbinned=48 000 → binned movie 48 GB → PCA adds ~48 GB → total ~96 GB  ✗ CRASH
#
#   For 1000×2000 frames (8 MB/frame):
#     nbinned= 5 000 → binned movie 40 GB → PCA adds ~40 GB → total ~80 GB  ⚠️ risky
#     nbinned= 3 000 → binned movie 24 GB → PCA adds ~24 GB → total ~48 GB  ✓ safe
#
#   GUIDANCE on a 96 GB machine:
#     512×512  : denoise=True is safe up to nbinned ≈ 40 000.
#     1000×2000: only enable if nbinned ≤ 3 000.  With the default nbinned=5 000
#                the detection step alone uses 40 GB; adding PCA pushes to ~80 GB,
#                leaving very little headroom.  Lower nbinned first, then enable.
ops["denoise"] = False

### 2.5 ROI detection — Cellpose (anatomical)

In [ ]:
# ── Cellpose mode ─────────────────────────────────────────────────────────────

# Use Cellpose segmentation to define ROI masks.
# 0 = disabled (use suite2p's own functional detection, default).
# 1 = segment on max_proj / meanImg
# 2 = segment on meanImg only
# 3 = segment on enhanced meanImg (high-pass filtered)
# 4 = segment on max_proj only
# ⚠️ HIDDEN from main docs page.  Default: 0.
ops["anatomical_only"] = 0

# Expected cell diameter in pixels for Cellpose.  0 = let Cellpose estimate it.
# This is the CELLPOSE diameter, different from suite2p's spatial_scale.
# Default: 0.
ops["diameter"] = 0

# Cellpose cell probability threshold.  Higher = stricter (fewer cells).
# ⚠️ HIDDEN — not in official docs.  Default: 0.0.
ops["cellprob_threshold"] = 0.0

# Cellpose flow error threshold.  Higher = more permissive (more cells accepted).
# ⚠️ HIDDEN — not in official docs.  Default: 0.4.
ops["flow_threshold"] = 0.4

# Apply a spatial high-pass filter to the image before Cellpose, expressed as a
# multiple of the diameter.  0 = no filtering.  Helps in dense neuropil.
# ⚠️ HIDDEN — not in official docs.  Default: 0.
ops["spatial_hp_cp"] = 0

# Cellpose model to use.  Can be a built-in model name ('cpsam', 'cyto3',
# 'cyto2', 'nuclei', …) or a path to a custom trained model.
# 'cpsam' = Cellpose SAM (Segment Anything), the current default.
# ⚠️ HIDDEN — not in official docs.  Default: 'cpsam'.
ops["pretrained_model"] = "cpsam"

### 2.6 Classifier (pre-classification during detection)

In [ ]:
# ── Pre-classification ────────────────────────────────────────────────────────

# Apply the cell classifier DURING detection.  ROIs with probability below this
# value are discarded early, before full signal extraction — speeds up the
# pipeline when there are many non-cell ROIs.  Set to 0.0 to disable.
# Default: 0.0.
ops["preclassify"] = 0.0

# ── Post-extraction classification ───────────────────────────────────────────

# Use suite2p's built-in logistic-regression classifier (trained on mouse V1
# recordings) to label ROIs as cell / non-cell after extraction.
# NOTE: Default is False in source — many tutorials incorrectly say True.
ops["use_builtin_classifier"] = False

# Path to a custom classifier file (.npy) trained via the suite2p GUI.
# Leave as "" to use the built-in model when use_builtin_classifier is True.
ops["classifier_path"] = ""

# Crop dendrites for classifier statistics like compactness.  Set True (default)
# to get cleaner soma-only shape statistics; False to include dendrites.
# Default: True.
ops["soma_crop"] = True

### 2.7 Signal extraction & neuropil

In [ ]:
# ── Neuropil extraction ───────────────────────────────────────────────────────

# Whether to compute and save Fneu (neuropil fluorescence).
# False = Fneu is set to zero for all cells.  Default: True.
ops["neuropil_extract"] = True

# Gap between the edge of the cell mask and the inner edge of the neuropil
# sampling annulus, in pixels.  Increase if you see somatic contamination in Fneu.
# Default: 2.
ops["inner_neuropil_radius"] = 2

# Minimum number of pixels in the neuropil annulus.  Cells near the image edge
# that have fewer available pixels use a larger annulus.  Default: 350.
ops["min_neuropil_pixels"] = 350

# Percentile of pixel weights (lam) within the cell area to use as a cutoff:
# pixels below this percentile are EXCLUDED from neuropil estimation to prevent
# dim cell pixels contaminating the neuropil signal.
# ⚠️ HIDDEN — not in official docs.  Default: 50.0.
ops["lam_percentile"] = 50.0

# When two ROIs share pixels, include those pixels in both ROIs' traces.
# False (default) = exclude shared pixels from both.  Default: False.
ops["allow_overlap"] = False

# ── Neuropil coefficient ──────────────────────────────────────────────────────

# Neuropil contamination fraction:
#   F_corrected = F_cell − neucoeff × F_neuropil
# Empirically estimated as 0.65–0.75 for GCaMP6 (Kerlin et al. 2010).
# Set to 0.0 to skip neuropil subtraction entirely.  Default: 0.7.
ops["neucoeff"] = 0.7

# ── Second channel (red cells) ────────────────────────────────────────────────

# Threshold for calling an ROI detected on the second (anatomical) channel.
# suite2p computes the ratio F_chan2 / F_chan1; ROIs above this are red-positive.
# Only relevant when nchannels == 2.  Default: 0.65.
ops["chan2_thres"] = 0.65

### 2.8 Spike deconvolution

In [ ]:
# ── Sensor time constant ──────────────────────────────────────────────────────

# GCaMP decay time constant (seconds). OASIS uses this to build its kernel;
# the kernel is NOT fitted to the data, so setting this correctly matters.
# Recommended values:
#   GCaMP6f  → 0.7 s
#   GCaMP6m  → 1.0 s  (default)
#   GCaMP6s  → 1.25–1.5 s
#   jGCaMP7f → ~0.5 s
#   jGCaMP8  → 0.2–0.4 s
# Default: 1.0.
ops["tau"] = 1.0

# ── Baseline estimation ───────────────────────────────────────────────────────

# Method used to estimate the fluorescence baseline (F0) before deconvolution.
#
# 'maximin' (default): robust rolling baseline.
#   Step 1 – Gaussian-smooth the trace with sigma = sig_baseline × fs.
#   Step 2 – compute the minimum of a rolling window of width win_baseline × fs.
#   Step 3 – compute the maximum of that rolling minimum.
#   This baseline tracks slow drifts while ignoring transient calcium events.
#
# 'constant': Gaussian-smooth with sig_baseline × fs, then take the minimum
#   of the whole smoothed trace.  Simple but doesn't track slow bleaching.
#
# 'constant_prctile': take the prctile_baseline percentile of the whole trace.
#   Fastest but most affected by very active cells.
ops["baseline"] = "maximin"

# Rolling window length in seconds for the maximin baseline computation.
# Should be longer than the longest expected calcium transient.  Default: 60.0 s.
ops["win_baseline"] = 60.0

# Gaussian smoothing sigma (seconds) applied to the trace before baseline
# estimation (used in both 'maximin' and 'constant' methods).  Default: 10.0 s.
ops["sig_baseline"] = 10.0

# Percentile used when baseline = 'constant_prctile'.
# 8th percentile ≈ low, robust estimate of baseline.  Default: 8.0.
ops["prctile_baseline"] = 8.0

### 2.9 Multi-plane & output

In [ ]:
# After processing each plane separately, merge all planes into a single
# 'suite2p/combined' folder so the GUI can display them together.
# Default: True.
ops["combined"] = True

### 2.10 Review — corrected defaults summary

The table below lists parameters where the **official docs are WRONG**:

In [ ]:
wrong_in_docs = [
    ("nimg_init",          "200 (docs)",   "300 (actual source)"),
    ("batch_size",         "200 (docs)",   "500 (actual source)"),
    ("keep_movie_raw",     "True (docs)",  "False (actual source)"),
    ("threshold_scaling",  "5.0 (docs)",   "1.0 (actual source)"),
    ("sparse_mode",        "False (docs)", "True (actual source)"),
    ("use_builtin_classifier", "True (many tutorials)", "False (actual source)"),
]
print(f"{'Parameter':<30} {'Docs say':<25} {'Source says'}")
print("-" * 75)
for name, wrong, right in wrong_in_docs:
    print(f"{name:<30} {wrong:<25} {right}")

In [ ]:
# Print and optionally verify all keys are present
ref = default_ops()
set_keys = set(ops.keys())
ref_keys = set(ref.keys())
missing  = ref_keys - set_keys
extra    = set_keys - ref_keys
print(f"Keys in default_ops() : {len(ref_keys)}")
print(f"Keys set in this notebook: {len(set_keys)}")
if missing: print(f"⚠️  Keys in default_ops not set here: {missing}")
if extra:   print(f"ℹ️  Extra keys set here (custom): {extra}")
if not missing and not extra: print("✓ All default_ops keys are covered.")

## 3. Run the pipeline

In [ ]:
# 'db' overrides ops for a specific dataset without permanently modifying ops.
# This is the recommended pattern when processing multiple sessions in a loop:
# keep ops as a template and pass session-specific paths in db.
db = {
    "data_path": DATA_PATH,
    "save_path0": SAVE_PATH,
    "fast_disk":  FAST_DISK,
}

output_ops = run_s2p(ops=ops, db=db)
print("\n✓ suite2p finished!")
print(f"  Results: {output_ops['save_path']}")

## 4. Load and inspect results

In [ ]:
plane_idx = 0
plane_dir = Path(SAVE_PATH) / "suite2p" / f"plane{plane_idx}"

ops_out = np.load(plane_dir / "ops.npy",    allow_pickle=True).item()
F       = np.load(plane_dir / "F.npy",      allow_pickle=True)
Fneu    = np.load(plane_dir / "Fneu.npy",   allow_pickle=True)
spks    = np.load(plane_dir / "spks.npy",   allow_pickle=True)
stat    = np.load(plane_dir / "stat.npy",   allow_pickle=True)
iscell  = np.load(plane_dir / "iscell.npy", allow_pickle=True)

cell_idx = np.where(iscell[:, 0].astype(bool))[0]

print(f"Plane      : {plane_idx}")
print(f"Total ROIs : {len(stat)}")
print(f"Cells      : {len(cell_idx)}")
print(f"Frames     : {F.shape[1]}")
print(f"Frame rate : {ops_out['fs']:.2f} Hz")
print(f"Duration   : {F.shape[1] / ops_out['fs']:.1f} s")

## 5. Quality-control plots

In [ ]:
clip = lambda img: dict(vmin=np.percentile(img, 1), vmax=np.percentile(img, 99))

# ── Mean & enhanced mean images ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(ops_out["meanImg"], cmap="gray", **clip(ops_out["meanImg"]))
axes[0].set_title("Mean image"); axes[0].axis("off")
if "meanImgE" in ops_out:
    axes[1].imshow(ops_out["meanImgE"], cmap="gray", **clip(ops_out["meanImgE"]))
    axes[1].set_title("Enhanced mean"); axes[1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── Registration shifts ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
t = np.arange(len(ops_out["yoff"])) / ops_out["fs"]
axes[0].plot(t, ops_out["yoff"], lw=0.5); axes[0].set_ylabel("Y shift (px)")
axes[0].set_title("Rigid registration shifts")
axes[1].plot(t, ops_out["xoff"], lw=0.5, color="C1"); axes[1].set_ylabel("X shift (px)")
axes[1].set_xlabel("Time (s)")
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell centroids on mean image ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(ops_out["meanImg"], cmap="gray", **clip(ops_out["meanImg"]))
for s in stat[cell_idx]:
    ax.scatter(s["med"][1], s["med"][0], s=5, c="cyan", linewidths=0)
ax.set_title(f"{len(cell_idx)} cells"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── ΔF/F traces ───────────────────────────────────────────────────────────────
N_SHOW = min(20, len(cell_idx))
show_idx = np.random.default_rng(0).choice(cell_idx, size=N_SHOW, replace=False)
F_corr = F[show_idx] - ops["neucoeff"] * Fneu[show_idx]
F0     = np.percentile(F_corr, 8, axis=1, keepdims=True)
dFF    = (F_corr - F0) / (F0 + 1)
t = np.arange(F.shape[1]) / ops_out["fs"]
fig, ax = plt.subplots(figsize=(16, N_SHOW * 0.5 + 1))
for i, trace in enumerate(dFF):
    ax.plot(t, trace + i * 3, lw=0.6, color="k")
ax.set_xlabel("Time (s)"); ax.set_ylabel("Cell (ΔF/F)"); ax.set_yticks([])
ax.set_title(f"ΔF/F traces ({N_SHOW} cells)")
plt.tight_layout(); plt.show()

In [ ]:
# ── Classifier score distribution ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(iscell[:, 1], bins=50, edgecolor="k", linewidth=0.3)
ax.axvline(0.5, color="red", linestyle="--", label="default threshold = 0.5")
ax.set_xlabel("Classifier probability (cell)"); ax.set_ylabel("Count")
ax.set_title("ROI classifier scores"); ax.legend()
plt.tight_layout(); plt.show()

## 6. Save ops for reproducibility

In [ ]:
def _ser(v):
    if hasattr(v, "tolist"): return v.tolist()
    if isinstance(v, Path): return str(v)
    return v

json_path = Path(SAVE_PATH) / "suite2p_ops_used.json"
with open(json_path, "w") as f:
    json.dump({k: _ser(v) for k, v in ops.items()}, f, indent=2, default=str)
print(f"Ops saved to: {json_path}")

---
## Appendix A: Hidden parameters quick reference

These keys exist in `default_ops()` but are **not documented** on the official settings page:

| Parameter | Default | What it does |
|-----------|---------|-------------|
| `force_sktiff` | `False` | Use scikit-image TIFF reader instead of ScanImage reader |
| `bruker_bidirectional` | `False` | Bruker bidirectional multiplane ordering (0,1,2,2,1,0) vs sequential |
| `nwb_file` | `""` | Direct path to NWB file input |
| `nwb_driver` | `""` | HDF5 driver for NWB file |
| `nwb_series` | `""` | TwoPhotonSeries name inside NWB file |
| `save_folder` | `[]` | Custom name for the results subdirectory |
| `move_bin` | `False` | Move (not copy) binary from fast_disk to save_path when done |
| `multiplane_parallel` | `False` | Experimental server-mode parallel plane processing |
| `ignore_flyback` | `[]` | Frame indices to ignore at the start of each file (PMT settling) |
| `bidi_corrected` | `False` | Mark data as already bidi-corrected by the acquisition software |
| `norm_frames` | `True` | Normalize frames before phase-correlation in registration |
| `th_badframes` | `1.0` | Threshold to flag high-motion frames (lower = more excluded) |
| `force_refImg` | `False` | Reuse a previously computed reference image stored in ops |
| `pad_fft` | `False` | Pad frame with zeros during FFT to reduce wrap-around artefacts |
| `spatial_hp_detect` | `25` | Spatial high-pass window (px) for neuropil subtraction before detection |
| `denoise` | `False` | PCA denoise the binned movie before sparse-mode detection |
| `anatomical_only` | `0` | Use Cellpose instead of suite2p's functional detection (1–4) |
| `diameter` | `0` | Expected cell diameter for Cellpose (0 = auto-estimate) |
| `cellprob_threshold` | `0.0` | Cellpose cell-probability threshold |
| `flow_threshold` | `0.4` | Cellpose flow error threshold |
| `spatial_hp_cp` | `0` | High-pass spatial filter for Cellpose input (multiples of diameter) |
| `pretrained_model` | `"cpsam"` | Cellpose model name or path to custom model |
| `lam_percentile` | `50.0` | Percentile of pixel weights below which pixels are excluded from neuropil |

## Appendix B: Output file reference

| File | Shape | Contents |
|------|-------|----------|
| `F.npy` | `(n_ROIs, n_frames)` | Raw fluorescence per ROI |
| `Fneu.npy` | `(n_ROIs, n_frames)` | Neuropil fluorescence per ROI |
| `spks.npy` | `(n_ROIs, n_frames)` | Deconvolved spike estimates (OASIS) |
| `iscell.npy` | `(n_ROIs, 2)` | `[:,0]` = is_cell flag; `[:,1]` = classifier probability |
| `stat.npy` | list of dicts | Per-ROI morphological statistics |
| `ops.npy` | dict | Final ops with computed fields (meanImg, shifts, …) |
| `data.bin` | raw float32 | Registered movie |
| `data_raw.bin` | raw float32 | Unregistered movie (only if `keep_movie_raw=True`) |

## Appendix C: Common troubleshooting

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| Too few cells found | `threshold_scaling` too high | Decrease (default is already 1.0 — check `spatial_scale` instead) |
| Too many false ROIs | `threshold_scaling` too low | Increase (try 2.0–5.0) |
| ROIs too small / large | `spatial_scale` wrong | Set `spatial_scale` to match cell size; or use Cellpose with correct `diameter` |
| Wavy horizontal stripes | Bidirectional phase offset | Set `do_bidiphase = True` |
| Poor registration | Low SNR, large motion | Try `two_step_registration = True`, increase `nimg_init` |
| Neuropil contaminates traces | `inner_neuropil_radius` too small | Increase to 3–5 |
| Spikes too sparse / dense | `tau` wrong for indicator | Adjust `tau` to match your sensor's decay constant |